# 01 - Priprema podataka

Ova sveska prati skriptu `scripts/01_prepare_data.py`. Cilj je da se originalni Ling-Spam korpus ucita, kratko analizira, ocisti od duplikata i podeli na trening, validacioni i test skup.

Najvaznije pravilo: podela mora biti stratifikovana, da odnos `ham` i `spam` mejlova ostane slican u svim skupovima.

In [13]:
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

ROOT

PosixPath('/Users/markoraskovic/Desktop/ML projekat')

## Ucitavanje originalnog skupa

Ling-Spam je arhiva sa tekstualnim fajlovima. U `bare` verziji korpusa fajlovi cije ime pocinje sa `spmsg` predstavljaju spam, a svi ostali fajlovi predstavljaju ham.

In [14]:
from pathlib import Path
import tarfile

import pandas as pd
from sklearn.model_selection import train_test_split

RAW_PATH = ROOT / "data" / "raw" / "lingspam_public.tar.gz"
CLEAN_PATH = ROOT / "data" / "processed" / "lingspam_clean.csv"
MODELING_PATH = ROOT / "data" / "processed" / "lingspam_modeling.csv"
SPLIT_DIR = ROOT / "data" / "processed" / "splits"

rows = []

with tarfile.open(RAW_PATH, "r:gz") as archive:
    for member in archive.getmembers():
        is_text_file = member.isfile() and member.name.startswith("lingspam_public/bare/") and member.name.endswith(".txt")
        if not is_text_file:
            continue

        file_name = Path(member.name).name
        label = "spam" if file_name.startswith("spmsg") else "ham"

        extracted_file = archive.extractfile(member)
        message = extracted_file.read().decode("latin-1", errors="replace").strip()
        rows.append([label, message])

df = pd.DataFrame(rows, columns=["label", "message"])
df.head()

,label,message
0,ham,Subject: re : 2 . 882 s - > np np\n\n> date : ...
1,ham,Subject: s - > np + np\n\nthe discussion of s ...
2,ham,Subject: 2 . 882 s - > np np\n\n. . . for me i...
3,ham,"Subject: gent conference\n\n"" for the listserv..."
4,ham,Subject: query : causatives in korean\n\ncould...


## Osnovna analiza

Dodajemo nekoliko jednostavnih kolona koje pomazu da razumemo strukturu mejlova: duzina teksta, broj reci, broj cifara i slicno.

In [15]:
df["label_id"] = (df["label"] == "spam").astype(int)
df["char_count"] = df["message"].str.len()
df["word_count"] = df["message"].str.split().str.len()
df["digit_count"] = df["message"].str.count(r"\d")
df["has_currency"] = df["message"].str.contains(r"[$\u00a3\u20ac]", regex=True)
df["has_long_number"] = df["message"].str.contains(r"\d{5,}", regex=True)
df["normalized_message"] = df["message"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()

class_counts = df["label"].value_counts()
class_percentages = (df["label"].value_counts(normalize=True) * 100).round(2)
duplicate_count = df.duplicated(["label", "message"]).sum()

summary = pd.DataFrame({"broj_poruka": class_counts, "procenat": class_percentages})
display(summary)
print(f"Broj duplikata: {duplicate_count}")

df.groupby("label")[["char_count", "word_count", "digit_count"]].mean().round(2)

,broj_poruka,procenat
label,,
ham,2412,83.37
spam,481,16.63


Broj duplikata: 17


,char_count,word_count,digit_count
label,,,
ham,3164.94,634.83,64.98
spam,3807.72,912.77,78.87


## Uklanjanje duplikata

Duplikate uklanjamo pre podele podataka. Tako isti tekst ne moze da se pojavi i u treningu i u testu.

In [16]:
df_modeling = df.drop_duplicates(["label", "message"]).reset_index(drop=True)

print(f"Broj mejlova pre uklanjanja duplikata: {len(df)}")
print(f"Broj mejlova posle uklanjanja duplikata: {len(df_modeling)}")

Broj mejlova pre uklanjanja duplikata: 2893
Broj mejlova posle uklanjanja duplikata: 2876


## Stratifikovana podela

Koristimo odnos 70/15/15. Prvo odvajamo 70% za trening, zatim preostalih 30% delimo na validaciju i test.

In [17]:
train_df, temp_df = train_test_split(
    df_modeling,
    test_size=0.30,
    stratify=df_modeling["label"],
    random_state=42,
)

validation_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42,
)

split_summary = pd.DataFrame({
    "skup": ["train", "validation", "test"],
    "broj_poruka": [len(train_df), len(validation_df), len(test_df)],
    "ham": [sum(train_df["label"] == "ham"), sum(validation_df["label"] == "ham"), sum(test_df["label"] == "ham")],
    "spam": [sum(train_df["label"] == "spam"), sum(validation_df["label"] == "spam"), sum(test_df["label"] == "spam")],
})

split_summary

,skup,broj_poruka,ham,spam
0,train,2013,1685,328
1,validation,431,361,70
2,test,432,362,70


## Cuvanje pripremljenih podataka

Ove CSV fajlove zatim koriste ostale sveske i Python skripte.

In [18]:
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(CLEAN_PATH, index=False)
df_modeling.to_csv(MODELING_PATH, index=False)
train_df.to_csv(SPLIT_DIR / "train.csv", index=False)
validation_df.to_csv(SPLIT_DIR / "validation.csv", index=False)
test_df.to_csv(SPLIT_DIR / "test.csv", index=False)

print(f"Sacuvano: {CLEAN_PATH}")
print(f"Sacuvano: {MODELING_PATH}")
print(f"Sacuvani splitovi: {SPLIT_DIR}")

Sacuvano: /Users/markoraskovic/Desktop/ML projekat/data/processed/lingspam_clean.csv
Sacuvano: /Users/markoraskovic/Desktop/ML projekat/data/processed/lingspam_modeling.csv
Sacuvani splitovi: /Users/markoraskovic/Desktop/ML projekat/data/processed/splits
